In [ ]:
# ============================================
# CELL 3/3: build NOISE_MAP (noise-only |g_a - g_b| at fixed w)
# ============================================

import os, sys, json, time
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# -----------------------------
# CONFIG
# -----------------------------
RUN_DIR = "out/E4_clean_2k_adamw"
ITER = 1999

K_PAIRS = 512       # 1 pair = 2 backward passes
SEED = 123

COORD_BUDGET_PER_GROUP = 600_000
N_COORD_SEEDS = 3
SAMPLES_PER_GROUP = 5_000_000

FIG_DIRNAME = "figures_tempered"
OVERWRITE = False
COMPRESS = True

ENABLE_TF32 = False
USE_CUDNN_BENCHMARK = True

UPDATE_EVERY = 4
PRINT_EVERY = 32
# -----------------------------

run_dir = Path(RUN_DIR).resolve()
assert run_dir.exists(), f"RUN_DIR not found: {run_dir}"
fig_dir = run_dir / FIG_DIRNAME
fig_dir.mkdir(parents=True, exist_ok=True)

npz_path  = fig_dir / f"noise_map_iter{ITER:07d}.npz"
meta_path = fig_dir / f"noise_map_iter{ITER:07d}.meta.json"

print("[info] saving:", npz_path)

torch.manual_seed(SEED)
np.random.seed(SEED)
PHASE_SEED = {"early": 111, "mid": 222, "late": 333}

if npz_path.exists() and not OVERWRITE:
    print("[ok] exists, loading")
    z = np.load(npz_path, allow_pickle=False)
    noise_map = {k: np.asarray(z[k]) for k in z.files}
    print("[ok] loaded groups:", len(noise_map))
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this cell.")
    DEVICE = "cuda"
    torch.backends.cuda.matmul.allow_tf32 = bool(ENABLE_TF32)
    torch.backends.cudnn.allow_tf32 = bool(ENABLE_TF32)
    torch.set_float32_matmul_precision("highest" if not ENABLE_TF32 else "high")
    torch.backends.cudnn.benchmark = bool(USE_CUDNN_BENCHMARK)

    sys.path.insert(0, str(Path.cwd()))
    from model import GPT, GPTConfig

    cfg = json.load(open(run_dir / "config_resolved.json", "r", encoding="utf-8"))
    n_layer = int(cfg.get("n_layer", 12))
    n_head  = int(cfg.get("n_head", 12))
    n_embd  = int(cfg.get("n_embd", 768))
    block_size = int(cfg.get("block_size", 1024))
    bias = bool(cfg.get("bias", False))
    dropout = float(cfg.get("dropout", 0.0))
    batch_size = int(cfg.get("batch_size", 12))
    vocab_size = int(cfg.get("vocab_size", 50304))

    data_dir = Path(cfg.get("data_dir", "data/openwebtext"))
    data_dir = data_dir if data_dir.is_absolute() else (Path.cwd() / data_dir).resolve()

    model = GPT(GPTConfig(
        block_size=block_size, vocab_size=vocab_size,
        n_layer=n_layer, n_head=n_head, n_embd=n_embd,
        dropout=dropout, bias=bias,
    )).to(DEVICE)

    ckpt_path = run_dir / "checkpoints" / f"ckpt_iter{ITER:07d}.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)

    model = model.to(dtype=torch.float32)
    model.train()
    name2param = dict(model.named_parameters())

    train_bin = data_dir / "train.bin"
    assert train_bin.exists(), f"train.bin not found: {train_bin}"
    train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
    train_len = int(train_data.shape[0])
    assert train_len > block_size + 2

    def get_batch_fast(bs: int, T: int, device: str):
        ix = torch.randint(train_len - T - 1, (bs,), device="cpu")
        x = torch.stack([torch.from_numpy(train_data[i:i+T].astype(np.int64, copy=False)) for i in ix])
        y = torch.stack([torch.from_numpy(train_data[i+1:i+1+T].astype(np.int64, copy=False)) for i in ix])
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

    b0 = n_layer // 3
    b1 = 2 * n_layer // 3
    PHASE_LAYERS = {"early": list(range(0, b0)), "mid": list(range(b0, b1)), "late": list(range(b1, n_layer))}

    def ordered_groups():
        out = []
        for phase in ["early","mid","late"]:
            for comp in ["q","k","v","proj"]:
                out.append(f"attn_{phase}_{comp}")
            for comp in ["fc","proj"]:
                out.append(f"mlp_{phase}_{comp}")
        return out
    GROUPS = ordered_groups()

    def sample_indices(numel: int, m: int, rng_local: np.random.Generator) -> np.ndarray:
        return rng_local.choice(numel, size=min(m, numel), replace=False).astype(np.int64)

    def to_idx_t(idx_np: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(idx_np).to(device=DEVICE, dtype=torch.long)

    def build_attn_qkv(layers, block: str, total_samples: int, rng_local):
        assert block in ("q","k","v")
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.attn.c_attn.weight"
            p = name2param[pname]
            numel = int(p.numel())
            block_numel = numel // 3
            offset = {"q":0,"k":1,"v":2}[block] * block_numel
            idx = sample_indices(block_numel, per_layer, rng_local) + offset
            samps.append((pname, to_idx_t(idx)))
        return samps

    def build_param(layers, suffix, total_samples, rng_local):
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.{suffix}"
            p = name2param[pname]
            idx = sample_indices(int(p.numel()), per_layer, rng_local)
            samps.append((pname, to_idx_t(idx)))
        return samps

    group_samplers = {}
    for phase, layers in PHASE_LAYERS.items():
        for comp in ["q","k","v","proj"]:
            group_samplers[f"attn_{phase}_{comp}"] = []
        for comp in ["fc","proj"]:
            group_samplers[f"mlp_{phase}_{comp}"] = []
        for s in range(N_COORD_SEEDS):
            rng_s = np.random.default_rng(SEED + 10_000*s + PHASE_SEED[phase])
            group_samplers[f"attn_{phase}_q"].append(build_attn_qkv(layers, "q", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_k"].append(build_attn_qkv(layers, "k", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_v"].append(build_attn_qkv(layers, "v", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_proj"].append(build_param(layers, "attn.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_fc"].append(build_param(layers, "mlp.c_fc.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_proj"].append(build_param(layers, "mlp.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))

    used_param_names = sorted({pname for seedlist in group_samplers.values() for samps in seedlist for (pname, _) in samps})

    class Reservoir:
        def __init__(self, size: int, rng: np.random.Generator):
            self.size = int(size)
            self.rng = rng
            self.buf = np.empty((self.size,), dtype=np.float32)
            self.filled = 0
            self.seen = 0
            self.replaced = 0

        def update(self, x: np.ndarray):
            x = np.asarray(x, dtype=np.float32).reshape(-1)
            m = int(x.size)
            if m <= 0:
                return
            if self.filled < self.size:
                take = min(self.size - self.filled, m)
                self.buf[self.filled:self.filled+take] = x[:take]
                self.filled += take
                self.seen += take
                x = x[take:]
                m = int(x.size)
                if m <= 0:
                    return
            base = int(self.seen)
            idx = base + np.arange(1, m+1, dtype=np.int64)
            prob = self.size / idx.astype(np.float64)
            keep = (self.rng.random(m) < prob)
            nk = int(np.sum(keep))
            if nk > 0:
                repl = self.rng.integers(0, self.size, size=nk, endpoint=False)
                self.buf[repl] = x[keep]
                self.replaced += nk
            self.seen += m

        def to_array(self):
            return self.buf.copy() if self.filled >= self.size else self.buf[:self.filled].copy()

    reservoirs = {g: Reservoir(SAMPLES_PER_GROUP, np.random.default_rng(SEED + 999 + i)) for i, g in enumerate(GROUPS)}
    amp_ctx = lambda: nullcontext()

    it = range(K_PAIRS) if tqdm is None else tqdm(range(K_PAIRS), total=K_PAIRS, desc="pairs |g_a-g_b|", dynamic_ncols=True)

    for k in it:
        # pass A
        model.zero_grad(set_to_none=True)
        Xa, Ya = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, la = model(Xa, Ya)
        la.backward()

        flat_a = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_a[pname] = None if gg is None else gg.detach().flatten()

        vecA_cpu = {}
        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_a[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vec = torch.cat(chunks, dim=0)
                vecA_cpu[(gname, sidx)] = vec.detach().cpu().numpy().astype(np.float32, copy=False)

        # pass B
        model.zero_grad(set_to_none=True)
        Xb, Yb = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, lb = model(Xb, Yb)
        lb.backward()

        flat_b = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_b[pname] = None if gg is None else gg.detach().flatten()

        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_b[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vecB = torch.cat(chunks, dim=0).detach().cpu().numpy().astype(np.float32, copy=False)
                d = np.abs(vecA_cpu[(gname, sidx)] - vecB)
                reservoirs[gname].update(d)

        if tqdm is not None and (k % UPDATE_EVERY == 0 or k == K_PAIRS-1):
            fills = np.array([reservoirs[g].filled / SAMPLES_PER_GROUP for g in GROUPS])
            it.set_postfix(min=f"{fills.min()*100:4.1f}%", avg=f"{fills.mean()*100:4.1f}%")

    noise_map = {g: reservoirs[g].to_array() for g in GROUPS}
    if COMPRESS:
        np.savez_compressed(str(npz_path), **noise_map)
    else:
        np.savez(str(npz_path), **noise_map)

    meta = dict(kind="noise_map", run_dir=str(run_dir), iter=int(ITER), seed=int(SEED),
                k_pairs=int(K_PAIRS), samples_per_group=int(SAMPLES_PER_GROUP),
                coord_budget_per_group=int(COORD_BUDGET_PER_GROUP), n_coord_seeds=int(N_COORD_SEEDS),
                enable_tf32=bool(ENABLE_TF32), npz=str(npz_path))
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[saved]", npz_path, meta_path)

noise_map

[info] saving: /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/noise_map_iter0001999.npz
number of parameters: 123.59M


/home/coder/tmp/ipykernel_538798/706729587.py:92: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
pairs |g_a-g_b|: 100%|

[saved] /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/noise_map_iter0001999.npz /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/noise_map_iter0001999.meta.json


{'attn_early_q': array([1.49528205e-05, 1.46898619e-05, 2.10337748e-06, ...,
        5.52959573e-05, 1.12943017e-05, 1.73382577e-05],
       shape=(5000000,), dtype=float32),
 'attn_early_k': array([9.2209513e-05, 5.6265853e-05, 3.0579005e-05, ..., 6.4139487e-05,
        8.8767838e-06, 4.8114940e-05], shape=(5000000,), dtype=float32),
 'attn_early_v': array([2.0994610e-04, 4.2642190e-05, 1.8234630e-04, ..., 1.9244864e-05,
        3.8698074e-04, 1.4751256e-04], shape=(5000000,), dtype=float32),
 'attn_early_proj': array([7.6599652e-05, 5.0782657e-04, 8.0188649e-05, ..., 1.3585345e-04,
        7.7708857e-05, 5.5868214e-04], shape=(5000000,), dtype=float32),
 'mlp_early_fc': array([1.1235060e-05, 2.2187141e-05, 3.3188244e-06, ..., 5.0304443e-06,
        5.1423725e-05, 4.2682441e-06], shape=(5000000,), dtype=float32),
 'mlp_early_proj': array([7.1123897e-05, 3.7314647e-04, 3.1993471e-04, ..., 3.4373174e-05,
        4.8016780e-05, 2.0459641e-04], shape=(5000000,), dtype=float32),
 'attn_mid

In [1]:
# ============================================
# CELL 1/3: build GRAD_MAP  (|g|)
# ============================================

import os, sys, json, time
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

print(1)
# -----------------------------
# CONFIG
# -----------------------------
RUN_DIR = "out/E4_clean_2k_adamw"
ITER = 200

K_BATCHES = 1024
SEED = 123

COORD_BUDGET_PER_GROUP = 600_000
N_COORD_SEEDS = 3
SAMPLES_PER_GROUP = 5_000_000

FIG_DIRNAME = "figures_tempered"
OVERWRITE = False
COMPRESS = True

ENABLE_TF32 = False
USE_CUDNN_BENCHMARK = True

UPDATE_EVERY = 8
PRINT_EVERY = 64
# -----------------------------

run_dir = Path(RUN_DIR).resolve()
assert run_dir.exists(), f"RUN_DIR not found: {run_dir}"
fig_dir = run_dir / FIG_DIRNAME
fig_dir.mkdir(parents=True, exist_ok=True)

npz_path  = fig_dir / f"grad_map_iter{ITER:07d}.npz"
meta_path = fig_dir / f"grad_map_iter{ITER:07d}.meta.json"

print("[info] saving:", npz_path)

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
PHASE_SEED = {"early": 111, "mid": 222, "late": 333}

if npz_path.exists() and not OVERWRITE:
    print("[ok] exists, loading")
    z = np.load(npz_path, allow_pickle=False)
    grad_map = {k: np.asarray(z[k]) for k in z.files}
    print("[ok] loaded groups:", len(grad_map))
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this cell.")
    DEVICE = "cuda"
    torch.backends.cuda.matmul.allow_tf32 = bool(ENABLE_TF32)
    torch.backends.cudnn.allow_tf32 = bool(ENABLE_TF32)
    torch.set_float32_matmul_precision("highest" if not ENABLE_TF32 else "high")
    torch.backends.cudnn.benchmark = bool(USE_CUDNN_BENCHMARK)

    sys.path.insert(0, str(Path.cwd()))
    from model import GPT, GPTConfig

    cfg = json.load(open(run_dir / "config_resolved.json", "r", encoding="utf-8"))
    n_layer = int(cfg.get("n_layer", 12))
    n_head  = int(cfg.get("n_head", 12))
    n_embd  = int(cfg.get("n_embd", 768))
    block_size = int(cfg.get("block_size", 1024))
    bias = bool(cfg.get("bias", False))
    dropout = float(cfg.get("dropout", 0.0))
    batch_size = int(cfg.get("batch_size", 12))
    vocab_size = int(cfg.get("vocab_size", 50304))

    data_dir = Path(cfg.get("data_dir", "data/openwebtext"))
    data_dir = data_dir if data_dir.is_absolute() else (Path.cwd() / data_dir).resolve()

    model = GPT(GPTConfig(
        block_size=block_size, vocab_size=vocab_size,
        n_layer=n_layer, n_head=n_head, n_embd=n_embd,
        dropout=dropout, bias=bias,
    )).to(DEVICE)

    ckpt_path = run_dir / "checkpoints" / f"ckpt_iter{ITER:07d}.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)

    model = model.to(dtype=torch.float32)
    model.train()
    name2param = dict(model.named_parameters())

    train_bin = data_dir / "train.bin"
    assert train_bin.exists(), f"train.bin not found: {train_bin}"
    train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
    train_len = int(train_data.shape[0])
    assert train_len > block_size + 2

    def get_batch_fast(bs: int, T: int, device: str):
        ix = torch.randint(train_len - T - 1, (bs,), device="cpu")
        x = torch.stack([torch.from_numpy(train_data[i:i+T].astype(np.int64, copy=False)) for i in ix])
        y = torch.stack([torch.from_numpy(train_data[i+1:i+1+T].astype(np.int64, copy=False)) for i in ix])
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

    b0 = n_layer // 3
    b1 = 2 * n_layer // 3
    PHASE_LAYERS = {"early": list(range(0, b0)), "mid": list(range(b0, b1)), "late": list(range(b1, n_layer))}

    def ordered_groups():
        out = []
        for phase in ["early","mid","late"]:
            for comp in ["q","k","v","proj"]:
                out.append(f"attn_{phase}_{comp}")
            for comp in ["fc","proj"]:
                out.append(f"mlp_{phase}_{comp}")
        return out

    GROUPS = ordered_groups()

    def sample_indices(numel: int, m: int, rng_local: np.random.Generator) -> np.ndarray:
        return rng_local.choice(numel, size=min(m, numel), replace=False).astype(np.int64)

    def to_idx_t(idx_np: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(idx_np).to(device=DEVICE, dtype=torch.long)

    def build_attn_qkv(layers, block: str, total_samples: int, rng_local):
        assert block in ("q","k","v")
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.attn.c_attn.weight"
            p = name2param[pname]
            numel = int(p.numel())
            block_numel = numel // 3
            offset = {"q":0,"k":1,"v":2}[block] * block_numel
            idx = sample_indices(block_numel, per_layer, rng_local) + offset
            samps.append((pname, to_idx_t(idx)))
        return samps

    def build_param(layers, suffix, total_samples, rng_local):
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.{suffix}"
            p = name2param[pname]
            idx = sample_indices(int(p.numel()), per_layer, rng_local)
            samps.append((pname, to_idx_t(idx)))
        return samps

    group_samplers = {}
    for phase, layers in PHASE_LAYERS.items():
        for comp in ["q","k","v","proj"]:
            group_samplers[f"attn_{phase}_{comp}"] = []
        for comp in ["fc","proj"]:
            group_samplers[f"mlp_{phase}_{comp}"] = []
        for s in range(N_COORD_SEEDS):
            rng_s = np.random.default_rng(SEED + 10_000*s + PHASE_SEED[phase])
            group_samplers[f"attn_{phase}_q"].append(build_attn_qkv(layers, "q", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_k"].append(build_attn_qkv(layers, "k", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_v"].append(build_attn_qkv(layers, "v", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_proj"].append(build_param(layers, "attn.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_fc"].append(build_param(layers, "mlp.c_fc.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_proj"].append(build_param(layers, "mlp.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))

    used_param_names = sorted({pname for seedlist in group_samplers.values() for samps in seedlist for (pname, _) in samps})

    class Reservoir:
        def __init__(self, size: int, rng: np.random.Generator):
            self.size = int(size)
            self.rng = rng
            self.buf = np.empty((self.size,), dtype=np.float32)
            self.filled = 0
            self.seen = 0
            self.replaced = 0

        def update(self, x: np.ndarray):
            x = np.asarray(x, dtype=np.float32).reshape(-1)
            m = int(x.size)
            if m <= 0:
                return
            if self.filled < self.size:
                take = min(self.size - self.filled, m)
                self.buf[self.filled:self.filled+take] = x[:take]
                self.filled += take
                self.seen += take
                x = x[take:]
                m = int(x.size)
                if m <= 0:
                    return
            base = int(self.seen)
            idx = base + np.arange(1, m+1, dtype=np.int64)
            prob = self.size / idx.astype(np.float64)
            keep = (self.rng.random(m) < prob)
            nk = int(np.sum(keep))
            if nk > 0:
                repl = self.rng.integers(0, self.size, size=nk, endpoint=False)
                self.buf[repl] = x[keep]
                self.replaced += nk
            self.seen += m

        def to_array(self):
            return self.buf.copy() if self.filled >= self.size else self.buf[:self.filled].copy()

    reservoirs = {g: Reservoir(SAMPLES_PER_GROUP, np.random.default_rng(SEED + 999 + i)) for i, g in enumerate(GROUPS)}
    amp_ctx = lambda: nullcontext()

    it = range(K_BATCHES) if tqdm is None else tqdm(range(K_BATCHES), total=K_BATCHES, desc="batches |g|", dynamic_ncols=True)
    t0 = time.time()

    for k in it:
        model.zero_grad(set_to_none=True)
        X, Y = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, loss = model(X, Y)
        loss.backward()

        flat_cache = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_cache[pname] = None if gg is None else gg.detach().flatten()

        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_cache[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vec = torch.cat(chunks, dim=0)
                reservoirs[gname].update(vec.abs_().detach().cpu().numpy().astype(np.float32, copy=False))

        if tqdm is not None and (k % UPDATE_EVERY == 0 or k == K_BATCHES-1):
            fills = np.array([reservoirs[g].filled / SAMPLES_PER_GROUP for g in GROUPS])
            it.set_postfix(min=f"{fills.min()*100:4.1f}%", avg=f"{fills.mean()*100:4.1f}%")

        if (k+1) % PRINT_EVERY == 0:
            avg_seen = float(np.mean([reservoirs[g].seen for g in GROUPS]))
            print(f"[detail] batch {k+1}/{K_BATCHES}, avg_seen={avg_seen/1e6:.2f}M")

    grad_map = {g: reservoirs[g].to_array() for g in GROUPS}
    if COMPRESS:
        np.savez_compressed(str(npz_path), **grad_map)
    else:
        np.savez(str(npz_path), **grad_map)

    meta = dict(kind="grad_map", run_dir=str(run_dir), iter=int(ITER), seed=int(SEED),
                k_batches=int(K_BATCHES), samples_per_group=int(SAMPLES_PER_GROUP),
                coord_budget_per_group=int(COORD_BUDGET_PER_GROUP), n_coord_seeds=int(N_COORD_SEEDS),
                enable_tf32=bool(ENABLE_TF32), npz=str(npz_path))
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[saved]", npz_path, meta_path)

grad_map

1
[info] saving: /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/grad_map_iter0000200.npz
number of parameters: 123.59M


/home/coder/tmp/ipykernel_7733/3280475785.py:94: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
batches |g|:   6%|▋    

[detail] batch 64/1024, avg_seen=115.20M


batches |g|:  12%|█▎        | 128/1024 [01:40<10:20,  1.44it/s, avg=100.0%, min=100.0%]

[detail] batch 128/1024, avg_seen=230.40M


...

batches |g|:  94%|█████████▍| 960/1024 [14:58<00:57,  1.11it/s, avg=100.0%, min=100.0%]

[detail] batch 960/1024, avg_seen=1728.00M


batches |g|: 100%|██████████| 1024/1024 [15:56<00:00,  1.07it/s, avg=100.0%, min=100.0%]


[detail] batch 1024/1024, avg_seen=1843.20M
[saved] /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/grad_map_iter0000200.npz /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/grad_map_iter0000200.meta.json


{'attn_early_q': array([9.7888042e-06, 6.8208110e-06, 2.5414643e-05, ..., 2.5618361e-05,
        5.6250860e-06, 2.2740858e-06], shape=(5000000,), dtype=float32),
 'attn_early_k': array([4.7927824e-06, 1.9506447e-06, 2.5013187e-06, ..., 1.2513003e-05,
        6.9679481e-06, 9.9542412e-06], shape=(5000000,), dtype=float32),
 'attn_early_v': array([2.8635608e-05, 6.6395281e-05, 3.1698386e-05, ..., 1.5316093e-05,
        1.0399077e-05, 1.4292923e-06], shape=(5000000,), dtype=float32),
 'attn_early_proj': array([8.0055004e-05, 1.2447276e-04, 2.1377406e-05, ..., 2.7799507e-04,
        3.2127340e-05, 4.1744031e-04], shape=(5000000,), dtype=float32),
 'mlp_early_fc': array([2.4787810e-06, 4.0631487e-05, 9.4637842e-05, ..., 4.1277468e-05,
        1.6290385e-05, 4.4858942e-05], shape=(5000000,), dtype=float32),
 'mlp_early_proj': array([3.9198476e-05, 1.5471762e-04, 4.6549299e-05, ..., 1.4707175e-04,
        2.4376271e-04, 9.4515541e-05], shape=(5000000,), dtype=float32),
 'attn_mid_q': array([6

In [2]:
# ============================================
# CELL 2/3: build DELTA_MAP (rolling |g_t - g_{t-1}|)
# ============================================

import os, sys, json, time
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import torch

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# -----------------------------
# CONFIG
# -----------------------------
RUN_DIR = "out/E4_clean_2k_adamw"
ITER = 200

K_BATCHES = 1024
SEED = 123

COORD_BUDGET_PER_GROUP = 600_000
N_COORD_SEEDS = 3
SAMPLES_PER_GROUP = 5_000_000

FIG_DIRNAME = "figures_tempered"
OVERWRITE = False
COMPRESS = True

ENABLE_TF32 = False
USE_CUDNN_BENCHMARK = True

UPDATE_EVERY = 8
PRINT_EVERY = 64
# -----------------------------

run_dir = Path(RUN_DIR).resolve()
assert run_dir.exists(), f"RUN_DIR not found: {run_dir}"
fig_dir = run_dir / FIG_DIRNAME
fig_dir.mkdir(parents=True, exist_ok=True)

npz_path  = fig_dir / f"delta_map_iter{ITER:07d}.npz"
meta_path = fig_dir / f"delta_map_iter{ITER:07d}.meta.json"

print("[info] saving:", npz_path)

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
PHASE_SEED = {"early": 111, "mid": 222, "late": 333}

if npz_path.exists() and not OVERWRITE:
    print("[ok] exists, loading")
    z = np.load(npz_path, allow_pickle=False)
    delta_map = {k: np.asarray(z[k]) for k in z.files}
    print("[ok] loaded groups:", len(delta_map))
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this cell.")
    DEVICE = "cuda"
    torch.backends.cuda.matmul.allow_tf32 = bool(ENABLE_TF32)
    torch.backends.cudnn.allow_tf32 = bool(ENABLE_TF32)
    torch.set_float32_matmul_precision("highest" if not ENABLE_TF32 else "high")
    torch.backends.cudnn.benchmark = bool(USE_CUDNN_BENCHMARK)

    sys.path.insert(0, str(Path.cwd()))
    from model import GPT, GPTConfig

    cfg = json.load(open(run_dir / "config_resolved.json", "r", encoding="utf-8"))
    n_layer = int(cfg.get("n_layer", 12))
    n_head  = int(cfg.get("n_head", 12))
    n_embd  = int(cfg.get("n_embd", 768))
    block_size = int(cfg.get("block_size", 1024))
    bias = bool(cfg.get("bias", False))
    dropout = float(cfg.get("dropout", 0.0))
    batch_size = int(cfg.get("batch_size", 12))
    vocab_size = int(cfg.get("vocab_size", 50304))

    data_dir = Path(cfg.get("data_dir", "data/openwebtext"))
    data_dir = data_dir if data_dir.is_absolute() else (Path.cwd() / data_dir).resolve()

    model = GPT(GPTConfig(
        block_size=block_size, vocab_size=vocab_size,
        n_layer=n_layer, n_head=n_head, n_embd=n_embd,
        dropout=dropout, bias=bias,
    )).to(DEVICE)

    ckpt_path = run_dir / "checkpoints" / f"ckpt_iter{ITER:07d}.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    model.load_state_dict(ckpt["model"], strict=True)

    model = model.to(dtype=torch.float32)
    model.train()
    name2param = dict(model.named_parameters())

    train_bin = data_dir / "train.bin"
    assert train_bin.exists(), f"train.bin not found: {train_bin}"
    train_data = np.memmap(train_bin, dtype=np.uint16, mode="r")
    train_len = int(train_data.shape[0])
    assert train_len > block_size + 2

    def get_batch_fast(bs: int, T: int, device: str):
        ix = torch.randint(train_len - T - 1, (bs,), device="cpu")
        x = torch.stack([torch.from_numpy(train_data[i:i+T].astype(np.int64, copy=False)) for i in ix])
        y = torch.stack([torch.from_numpy(train_data[i+1:i+1+T].astype(np.int64, copy=False)) for i in ix])
        return x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)

    b0 = n_layer // 3
    b1 = 2 * n_layer // 3
    PHASE_LAYERS = {"early": list(range(0, b0)), "mid": list(range(b0, b1)), "late": list(range(b1, n_layer))}

    def ordered_groups():
        out = []
        for phase in ["early","mid","late"]:
            for comp in ["q","k","v","proj"]:
                out.append(f"attn_{phase}_{comp}")
            for comp in ["fc","proj"]:
                out.append(f"mlp_{phase}_{comp}")
        return out
    GROUPS = ordered_groups()

    def sample_indices(numel: int, m: int, rng_local: np.random.Generator) -> np.ndarray:
        return rng_local.choice(numel, size=min(m, numel), replace=False).astype(np.int64)

    def to_idx_t(idx_np: np.ndarray) -> torch.Tensor:
        return torch.from_numpy(idx_np).to(device=DEVICE, dtype=torch.long)

    def build_attn_qkv(layers, block: str, total_samples: int, rng_local):
        assert block in ("q","k","v")
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.attn.c_attn.weight"
            p = name2param[pname]
            numel = int(p.numel())
            block_numel = numel // 3
            offset = {"q":0,"k":1,"v":2}[block] * block_numel
            idx = sample_indices(block_numel, per_layer, rng_local) + offset
            samps.append((pname, to_idx_t(idx)))
        return samps

    def build_param(layers, suffix, total_samples, rng_local):
        samps = []
        per_layer = max(1, total_samples // max(1, len(layers)))
        for li in layers:
            pname = f"transformer.h.{li}.{suffix}"
            p = name2param[pname]
            idx = sample_indices(int(p.numel()), per_layer, rng_local)
            samps.append((pname, to_idx_t(idx)))
        return samps

    group_samplers = {}
    for phase, layers in PHASE_LAYERS.items():
        for comp in ["q","k","v","proj"]:
            group_samplers[f"attn_{phase}_{comp}"] = []
        for comp in ["fc","proj"]:
            group_samplers[f"mlp_{phase}_{comp}"] = []
        for s in range(N_COORD_SEEDS):
            rng_s = np.random.default_rng(SEED + 10_000*s + PHASE_SEED[phase])
            group_samplers[f"attn_{phase}_q"].append(build_attn_qkv(layers, "q", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_k"].append(build_attn_qkv(layers, "k", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_v"].append(build_attn_qkv(layers, "v", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"attn_{phase}_proj"].append(build_param(layers, "attn.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_fc"].append(build_param(layers, "mlp.c_fc.weight", COORD_BUDGET_PER_GROUP, rng_s))
            group_samplers[f"mlp_{phase}_proj"].append(build_param(layers, "mlp.c_proj.weight", COORD_BUDGET_PER_GROUP, rng_s))

    used_param_names = sorted({pname for seedlist in group_samplers.values() for samps in seedlist for (pname, _) in samps})

    class Reservoir:
        def __init__(self, size: int, rng: np.random.Generator):
            self.size = int(size)
            self.rng = rng
            self.buf = np.empty((self.size,), dtype=np.float32)
            self.filled = 0
            self.seen = 0
            self.replaced = 0

        def update(self, x: np.ndarray):
            x = np.asarray(x, dtype=np.float32).reshape(-1)
            m = int(x.size)
            if m <= 0:
                return
            if self.filled < self.size:
                take = min(self.size - self.filled, m)
                self.buf[self.filled:self.filled+take] = x[:take]
                self.filled += take
                self.seen += take
                x = x[take:]
                m = int(x.size)
                if m <= 0:
                    return
            base = int(self.seen)
            idx = base + np.arange(1, m+1, dtype=np.int64)
            prob = self.size / idx.astype(np.float64)
            keep = (self.rng.random(m) < prob)
            nk = int(np.sum(keep))
            if nk > 0:
                repl = self.rng.integers(0, self.size, size=nk, endpoint=False)
                self.buf[repl] = x[keep]
                self.replaced += nk
            self.seen += m

        def to_array(self):
            return self.buf.copy() if self.filled >= self.size else self.buf[:self.filled].copy()

    reservoirs = {g: Reservoir(SAMPLES_PER_GROUP, np.random.default_rng(SEED + 999 + i)) for i, g in enumerate(GROUPS)}
    prev = {g: [None]*N_COORD_SEEDS for g in GROUPS}
    amp_ctx = lambda: nullcontext()

    it = range(K_BATCHES) if tqdm is None else tqdm(range(K_BATCHES), total=K_BATCHES, desc="batches |g_t-g_{t-1}|", dynamic_ncols=True)

    for k in it:
        model.zero_grad(set_to_none=True)
        X, Y = get_batch_fast(batch_size, block_size, DEVICE)
        with amp_ctx():
            _, loss = model(X, Y)
        loss.backward()

        flat_cache = {}
        for pname in used_param_names:
            gg = name2param[pname].grad
            flat_cache[pname] = None if gg is None else gg.detach().flatten()

        for gname in GROUPS:
            for sidx, samplers in enumerate(group_samplers[gname]):
                chunks = []
                for pname, idx_t in samplers:
                    flat = flat_cache[pname]
                    if flat is None:
                        chunks.append(torch.zeros((idx_t.numel(),), device=DEVICE, dtype=torch.float32))
                    else:
                        chunks.append(flat.index_select(0, idx_t))
                vec = torch.cat(chunks, dim=0)

                if prev[gname][sidx] is None:
                    prev[gname][sidx] = vec
                    continue
                d = (prev[gname][sidx] - vec).abs_()
                prev[gname][sidx] = vec
                reservoirs[gname].update(d.detach().cpu().numpy().astype(np.float32, copy=False))

        if tqdm is not None and (k % UPDATE_EVERY == 0 or k == K_BATCHES-1):
            fills = np.array([reservoirs[g].filled / SAMPLES_PER_GROUP for g in GROUPS])
            it.set_postfix(min=f"{fills.min()*100:4.1f}%", avg=f"{fills.mean()*100:4.1f}%")

    delta_map = {g: reservoirs[g].to_array() for g in GROUPS}
    if COMPRESS:
        np.savez_compressed(str(npz_path), **delta_map)
    else:
        np.savez(str(npz_path), **delta_map)

    meta = dict(kind="delta_map", run_dir=str(run_dir), iter=int(ITER), seed=int(SEED),
                k_batches=int(K_BATCHES), samples_per_group=int(SAMPLES_PER_GROUP),
                coord_budget_per_group=int(COORD_BUDGET_PER_GROUP), n_coord_seeds=int(N_COORD_SEEDS),
                enable_tf32=bool(ENABLE_TF32), npz=str(npz_path))
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[saved]", npz_path, meta_path)

delta_map

[info] saving: /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/delta_map_iter0000200.npz
number of parameters: 123.59M


/home/coder/tmp/ipykernel_7733/3462598939.py:93: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
batches |g_t-g_{t-1}|: 

[saved] /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/delta_map_iter0000200.npz /home/coder/project/dcbench-c1-sophia/out/E4_clean_2k_adamw/figures_tempered/delta_map_iter0000200.meta.json


{'attn_early_q': array([1.8989436e-05, 4.8496249e-06, 4.0905594e-05, ..., 2.8787745e-06,
        9.2722894e-06, 6.4221804e-06], shape=(5000000,), dtype=float32),
 'attn_early_k': array([2.7618134e-06, 4.7873527e-06, 7.0310773e-05, ..., 6.0220736e-07,
        1.5890048e-05, 1.6248876e-05], shape=(5000000,), dtype=float32),
 'attn_early_v': array([4.9233859e-06, 2.7591144e-05, 2.1457772e-05, ..., 4.1480736e-05,
        1.0332715e-04, 1.1553653e-05], shape=(5000000,), dtype=float32),
 'attn_early_proj': array([3.9474544e-06, 1.4944373e-04, 7.1760616e-05, ..., 2.3255571e-04,
        2.6367616e-05, 1.6784694e-04], shape=(5000000,), dtype=float32),
 'mlp_early_fc': array([1.0302336e-05, 3.2997559e-05, 3.4804871e-05, ..., 9.5558535e-06,
        2.2120790e-05, 3.3419885e-05], shape=(5000000,), dtype=float32),
 'mlp_early_proj': array([4.9583985e-05, 5.3793029e-04, 8.2762264e-05, ..., 1.3780424e-04,
        6.0712569e-05, 4.7734364e-05], shape=(5000000,), dtype=float32),
 'attn_mid_q': array([1

In [ ]:
exit()

: 